# Practice Word2Vec Hard

## Word2Vec Model 2023

Write a program that trains Word2Vec model. Do not use print() instructions in your code, otherwise test procedure will not succeed; the message "Wrong Answer" indicates answer format is incorrect (print() in the code, missing words in the dictionary, etc.). The message "Embeddings are not good enough" means you're on the right track and you should focus on the model improvement.

You may think of the input string as being pre-processed with the following function:

```python
import re
import string

def clean(inp: str) -> str:

    inp = inp.translate(str.maketrans(string.punctuation, " "*len(string.punctuation)))

    inp = re.sub(r'\s+', ' ', inp.lower())

    return inp

```

I.e. given the input "Your string!" the output will be "your string ".


Input: data (string) - cleaned documents without punctuation in one line
Output: w2v_dict (dict: key (string) - a word from vocabulary, value (numpy array) - the word's embedding)

Time limit: 25 seconds
Memory limit: 128 MB

In [1]:
import numpy as np
from collections import Counter


def train(data: str):
    """
    Train a Word2Vec-style Skip-gram model with Negative Sampling.

    Parameters
    ----------
    data : str
        Preprocessed corpus. Words are lowercase and punctuation-free.

    Returns
    -------
    w2v_dict : dict
        word -> numpy.ndarray embedding
    """

    # ------------------------------------------------------------
    # 1. Tokenization and vocabulary construction
    # ------------------------------------------------------------
    words = data.split()

    if not words:
        return {}

    counts = Counter(words)

    # Keep every word in the vocabulary.
    vocab = list(counts.keys())
    word_to_id = {word: i for i, word in enumerate(vocab)}
    vocab_size = len(vocab)

    # Convert corpus to integer word IDs.
    token_ids = np.fromiter(
        (word_to_id[w] for w in words),
        dtype=np.int32,
        count=len(words)
    )

    # We no longer need the original token list.
    del words

    # ------------------------------------------------------------
    # 2. Hyperparameters
    # ------------------------------------------------------------
    # A 100-dimensional embedding is a good compromise between
    # semantic quality, speed and memory usage.
    dim = 100

    # Context window.
    window = 5

    # Number of negative samples for each positive pair.
    negative_samples = 5

    # Adaptive number of epochs.
    n_tokens = len(token_ids)

    if n_tokens < 50000:
        epochs = 12
    elif n_tokens < 200000:
        epochs = 7
    elif n_tokens < 500000:
        epochs = 4
    else:
        epochs = 2

    # ------------------------------------------------------------
    # 3. Random initialization
    # ------------------------------------------------------------
    rng = np.random.default_rng(42)

    # Input / center-word embeddings.
    W_in = (
        rng.standard_normal((vocab_size, dim), dtype=np.float32)
        * np.float32(0.5 / np.sqrt(dim))
    )

    # Output / context-word embeddings.
    W_out = np.zeros((vocab_size, dim), dtype=np.float32)

    # ------------------------------------------------------------
    # 4. Negative-sampling distribution
    # ------------------------------------------------------------
    #
    # Word2Vec does not sample negative words uniformly.
    # It uses approximately:
    #
    #       P(w) ∝ count(w)^0.75
    #
    # This gives frequent words a higher probability while
    # preventing them from dominating training.
    # ------------------------------------------------------------
    frequencies = np.asarray(
        [counts[w] for w in vocab],
        dtype=np.float64
    )

    negative_distribution = frequencies ** 0.75
    negative_distribution /= negative_distribution.sum()

    # Cumulative distribution allows fast sampling with
    # np.searchsorted.
    cumulative = np.cumsum(negative_distribution)

    # ------------------------------------------------------------
    # 5. Optional subsampling of very frequent words
    # ------------------------------------------------------------
    #
    # Very frequent words such as "the", "of", "and" provide
    # relatively little semantic information.
    #
    # We remove some occurrences during training, but never remove
    # the words from the final dictionary.
    # ------------------------------------------------------------
    if n_tokens >= 1000:
        total = float(n_tokens)
        freq = frequencies / total

        threshold = 1e-4

        keep_prob = np.minimum(
            1.0,
            (np.sqrt(freq / threshold) + 1.0)
            * (threshold / freq)
        )

        keep_prob = np.nan_to_num(
            keep_prob,
            nan=1.0,
            posinf=1.0,
            neginf=1.0
        )

        random_values = rng.random(n_tokens)
        keep_mask = random_values < keep_prob[token_ids]

        sampled_tokens = token_ids[keep_mask]

        # Do not allow aggressive subsampling to destroy a small corpus.
        if len(sampled_tokens) >= max(100, int(0.25 * n_tokens)):
            token_ids = sampled_tokens

    # ------------------------------------------------------------
    # 6. Training parameters
    # ------------------------------------------------------------
    initial_lr = 0.045
    min_lr = 0.003

    # Number of training examples processed in one vectorized step.
    batch_size = 256

    # ------------------------------------------------------------
    # 7. Helper: generate negative word IDs
    # ------------------------------------------------------------
    def sample_negative(size):
        r = rng.random(size)
        return np.searchsorted(cumulative, r).astype(np.int32)

    # ------------------------------------------------------------
    # 8. Skip-gram training
    # ------------------------------------------------------------
    #
    # For every:
    #
    #       center word -> context word
    #
    # create a positive training example.
    #
    # Then sample several random negative context words.
    #
    # Positive:
    #       (center, context) -> 1
    #
    # Negative:
    #       (center, random_word) -> 0
    #
    # This is exactly the basic idea of Skip-gram with
    # Negative Sampling.
    # ------------------------------------------------------------

    current_lr = initial_lr

    for epoch in range(epochs):

        # Shuffle the corpus for every epoch.
        order = rng.permutation(len(token_ids))
        corpus = token_ids[order]

        # Generate training pairs for each distance from the center.
        for offset in range(1, window + 1):

            if len(corpus) <= offset:
                break

            # Left/right context.
            centers_left = corpus[:-offset]
            contexts_right = corpus[offset:]

            centers_right = corpus[offset:]
            contexts_left = corpus[:-offset]

            # Train both directions:
            #
            # word_i -> word_(i+offset)
            # word_(i+offset) -> word_i
            #
            centers = np.concatenate(
                (centers_left, centers_right)
            )

            contexts = np.concatenate(
                (contexts_right, contexts_left)
            )

            # Shuffle the generated pairs.
            pair_order = rng.permutation(len(centers))

            for start in range(0, len(pair_order), batch_size):

                end = min(start + batch_size, len(pair_order))
                indices = pair_order[start:end]

                center_ids = centers[indices]
                positive_ids = contexts[indices]

                batch_n = len(center_ids)

                # ------------------------------------------------
                # Positive scores
                # ------------------------------------------------
                center_vecs = W_in[center_ids]
                positive_vecs = W_out[positive_ids]

                positive_scores = np.sum(
                    center_vecs * positive_vecs,
                    axis=1
                )

                # sigmoid(score)
                positive_sigmoid = 1.0 / (
                    1.0 + np.exp(
                        -np.clip(positive_scores, -15.0, 15.0)
                    )
                )

                # dL/dscore for positive examples:
                #
                # sigmoid(score) - 1
                #
                positive_gradient = positive_sigmoid - 1.0

                # ------------------------------------------------
                # Negative samples
                # ------------------------------------------------
                negative_ids = sample_negative(
                    (batch_n, negative_samples)
                )

                negative_vecs = W_out[negative_ids]

                negative_scores = np.einsum(
                    "bd,bkd->bk",
                    center_vecs,
                    negative_vecs
                )

                negative_sigmoid = 1.0 / (
                    1.0 + np.exp(
                        -np.clip(negative_scores, -15.0, 15.0)
                    )
                )

                # For negative examples:
                #
                # dL/dscore = sigmoid(score)
                #
                negative_gradient = negative_sigmoid

                # ------------------------------------------------
                # Gradient for input vectors
                # ------------------------------------------------
                grad_center = (
                    positive_gradient[:, None]
                    * positive_vecs
                )

                grad_center += np.sum(
                    negative_gradient[:, :, None]
                    * negative_vecs,
                    axis=1
                )

                # ------------------------------------------------
                # Gradient for positive output vectors
                # ------------------------------------------------
                grad_positive = (
                    positive_gradient[:, None]
                    * center_vecs
                )

                # ------------------------------------------------
                # Gradient for negative output vectors
                # ------------------------------------------------
                grad_negative = (
                    negative_gradient[:, :, None]
                    * center_vecs[:, None, :]
                )

                # ------------------------------------------------
                # Update input embeddings
                # ------------------------------------------------
                np.add.at(
                    W_in,
                    center_ids,
                    -current_lr * grad_center
                )

                # ------------------------------------------------
                # Update positive output embeddings
                # ------------------------------------------------
                np.add.at(
                    W_out,
                    positive_ids,
                    -current_lr * grad_positive
                )

                # ------------------------------------------------
                # Update negative output embeddings
                # ------------------------------------------------
                np.add.at(
                    W_out,
                    negative_ids.reshape(-1),
                    -current_lr
                    * grad_negative.reshape(-1, dim)
                )

        # Linear learning-rate decay.
        current_lr = max(
            min_lr,
            initial_lr
            * (1.0 - float(epoch + 1) / float(epochs))
        )

    # ------------------------------------------------------------
    # 9. Construct final embeddings
    # ------------------------------------------------------------
    #
    # Word2Vec normally uses W_in as the word representation.
    #
    # Averaging W_in and W_out can sometimes produce slightly
    # smoother similarity structure.
    # ------------------------------------------------------------
    embeddings = W_in + W_out

    # L2-normalization makes cosine similarity equivalent to
    # a simple dot product and is useful for embedding evaluation.
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)

    embeddings = embeddings / np.maximum(norms, 1e-8)

    # ------------------------------------------------------------
    # 10. Return ALL vocabulary words
    # ------------------------------------------------------------
    w2v_dict = {
        word: embeddings[i]
        for i, word in enumerate(vocab)
    }

    return w2v_dict

In [6]:
inputs = """
cat dog cat dog cat dog cat dog cat dog
cat animal cat animal cat animal
dog animal dog animal dog animal
cat pet cat pet cat pet
dog pet dog pet dog pet
apple orange apple orange apple orange
apple fruit apple fruit apple fruit
orange fruit orange fruit orange fruit
car bus car bus car bus
car vehicle car vehicle car vehicle
bus vehicle bus vehicle bus vehicle
"""

result = train(inputs)

cat = result["cat"]
dog = result["dog"]
apple = result["apple"]
orange = result["orange"]
car = result["car"]
bus = result["bus"]

print("cat-dog:", np.dot(cat, dog))
print("apple-orange:", np.dot(apple, orange))
print("car-bus:", np.dot(car, bus))

cat-dog: 0.19149083
apple-orange: -0.47278377
car-bus: 0.622465
